In [4]:
import json
from collections import Counter, defaultdict
import pandas as pd
from pathlib import Path



In [7]:
DATA_PATH = "data/entities.ftm.json"
SAMPLE_SIZE = 100_000

file_path = Path(DATA_PATH)
# check data exists
if not file_path.is_file():
    print("Uh oh...")
print("Yes!")

Yes!


Profiling

In [8]:
schemas = Counter()
targets = Counter()
countries = Counter()
name_lengths = []
has_address = Counter()
has_alias = Counter()
has_identifier = Counter()
identifier_types = Counter()


try:
    with open(DATA_PATH, "r") as f:
        for i, line in enumerate(f):
            if i >= SAMPLE_SIZE:
                break
            try:
                entity = json.loads(line)
                schema = entity.get("schema", "unknown")
                schemas[schema] += 1
                targets[entity.get("target", False)] += 1
                
                props = entity.get("properties", {}) or {}
                
                for c in props.get("country", []):
                    countries[c] += 1
                
                for name in props.get("name", []):
                    name_lengths.append(len(name))
                
                has_address["yes" if props.get("address") else "no"] += 1
                has_alias["yes" if props.get("alias") else "no"] += 1
                
                id_keys = ["passportNumber", "taxNumber", "idNumber", "registrationNumber", "innCode", "ogrnCode", "npiCode", "leiCode"]
                found_id = False
                for key in id_keys:
                    if props.get(key):
                        found_id = True
                        identifier_types[key] += 1
                has_identifier["yes" if found_id else "no"] += 1
            except:
                continue
except FileNotFoundError:
    print(f"{DATA_PATH} does not exist. Please check again.")

print(f"Profiled {SAMPLE_SIZE:,} lines")

Profiled 100,000 lines


Schema distribution

In [9]:
df_schemas = pd.DataFrame(schemas.most_common(), columns=["schema", "count"])
df_schemas["pct"] = (df_schemas["count"] / df_schemas["count"].sum() * 100).round(1)
print(df_schemas.head(20).to_string(index=False))

        schema  count  pct
     Occupancy  53079 53.1
        Person  21689 21.7
       Company  10780 10.8
    Succession   4945  4.9
        Vessel   2041  2.0
   LegalEntity   1777  1.8
  Organization   1727  1.7
        Family   1226  1.2
     Ownership   1016  1.0
   UnknownLink    744  0.7
       Address    442  0.4
  Directorship    237  0.2
      Position    194  0.2
      Airplane     76  0.1
 Documentation     13  0.0
      Security      6  0.0
    Employment      3  0.0
Representation      2  0.0
    Membership      1  0.0
    PublicBody      1  0.0


Target flag

In [10]:
print(f"target=True (sanctioned/PEP): {targets.get(True, 0):,}")
print(f"target=False (related): {targets.get(False, 0):,}")

target=True (sanctioned/PEP): 34,310
target=False (related): 65,690


Top countries

In [11]:
df_countries = pd.DataFrame(countries.most_common(20), columns=["country", "count"])
print(df_countries.to_string(index=False))

country  count
     us  18143
     ru   3250
     cn   1473
     tr    467
     ir    443
     mx    339
     hk    336
     ae    335
     gb    243
     in    234
     ua    164
     de    159
     co    143
     br    134
     sg    125
     sy    125
     no    124
     pk    113
     iq    112
     eg    109


Field presence

In [12]:
print(f"Has address: {has_address.get('yes', 0):,} ({has_address.get('yes',0)/sum(has_address.values())*100:.1f}%)")
print(f"Has alias: {has_alias.get('yes', 0):,} ({has_alias.get('yes',0)/sum(has_alias.values())*100:.1f}%)")
print(f"Has identifier: {has_identifier.get('yes', 0):,} ({has_identifier.get('yes',0)/sum(has_identifier.values())*100:.1f}%)")

Has address: 26,749 (26.7%)
Has alias: 9,163 (9.2%)
Has identifier: 19,347 (19.3%)


Find entities with multiple addresses

In [13]:
multi_address_entities = []
persons_with_aliases = []
orgs_with_reg_numbers = []

with open(DATA_PATH, "r") as f:
    for i, line in enumerate(f):
        if i >= SAMPLE_SIZE:
            break
        try:
            entity = json.loads(line)
            schema = entity.get("schema")
            props = entity.get("properties", {}) or {}
            
            addrs = props.get("address", [])
            if len(addrs) >= 2 and schema in ["Person", "Organization", "Company", "LegalEntity"]:
                multi_address_entities.append({
                    "name": props.get("name", [""])[0],
                    "schema": schema,
                    "addresses": addrs,
                    "source_id": entity.get("id")
                })
            
            aliases = props.get("alias", [])
            if schema == "Person" and len(aliases) >= 2:
                persons_with_aliases.append({
                    "name": props.get("name", [""])[0],
                    "aliases": aliases,
                    "source_id": entity.get("id")
                })
            
            if schema in ["Organization", "Company", "LegalEntity"] and props.get("registrationNumber"):
                orgs_with_reg_numbers.append({
                    "name": props.get("name", [""])[0],
                    "reg": props.get("registrationNumber", [""])[0],
                    "source_id": entity.get("id")
                })
        except:
            continue

print(f"Entities with 2+ addresses: {len(multi_address_entities)}")
print(f"Persons with 2+ aliases: {len(persons_with_aliases)}")
print(f"Orgs with registration numbers: {len(orgs_with_reg_numbers)}")

Entities with 2+ addresses: 18949
Persons with 2+ aliases: 1237
Orgs with registration numbers: 3265


Show samples

In [14]:
print("=== ENTITIES WITH MULTIPLE ADDRESSES ===")
for e in multi_address_entities[:5]:
    print(f"\n{e['name']} ({e['schema']})")
    for a in e['addresses']:
        print(f"  - {a}")

print("\n=== PERSONS WITH ALIASES ===")
for p in persons_with_aliases[:5]:
    print(f"\n{p['name']}")
    for a in p['aliases']:
        print(f"  alias: {a}")

print("\n=== ORGS WITH REG NUMBERS ===")
for o in orgs_with_reg_numbers[:5]:
    print(f"{o['name']} | reg: {o['reg']}")

=== ENTITIES WITH MULTIPLE ADDRESSES ===

Myanmar Yatai International Holding Group Company Ltd (Company)
  - HPA-AN CITY
  - Shwe Kokko Village, Myawaddy Township, Karen State
  - Hpa-An City

Товариство з обмеженою відповідальністю "Зелінський Групп" (Company)
  - УЛ. ДУБИНИНСКАЯ Д. 57 СТР. 2, Г.МОСКВА, 115054
  - Російська Федерація, 115054, м. Москва, вул. Дубінинська, буд. 57 (будова 2, антр. 1, прим. III, кімн. 12А)

Open Joint-Stock Company "Elektrostal ChemicalMechanical Plant named after N.D.Zelinsky" (Company)
  - УЛ. КАРЛА МАРКСА Д.1, Г. ЭЛЕКТРОСТАЛЬ, МОСКОВСКАЯ ОБЛАСТЬ, 144001
  - Російська Федерація, 144001, Московська обл., м. Електросталь, вул. Карла Маркса, буд. 1
  - Electrostal, Karl Marx St., 1, Moscow, Russian Federation, 144001
  - Karl Marx St., 1, Electrostal, RUSSIAN FEDERATION, 144001

Приватне підприємство "Магістар-СГ" (Company)
  - Україна, 79034, Львівська обл., місто Львів, ВУЛИЦЯ НАВРОЦЬКОГО, будинок 33
  - Україна, 79034, м. Львів, вул. Навроцького, буд.

Find address cluster

In [15]:
address_to_entities = defaultdict(list)

with open(DATA_PATH, "r") as f:
    for i, line in enumerate(f):
        if i >= SAMPLE_SIZE:
            break
        try:
            entity = json.loads(line)
            schema = entity.get("schema")
            if schema not in ["Person", "Organization", "Company", "LegalEntity"]:
                continue
            props = entity.get("properties", {}) or {}
            for addr in props.get("address", []):
                name = props.get("name", [""])[0]
                address_to_entities[addr].append({
                    "name": name,
                    "schema": schema,
                    "source_id": entity.get("id")
                })
        except:
            continue

shared = {addr: ents for addr, ents in address_to_entities.items() if len(ents) >= 2}
print(f"Addresses shared by 2+ entities: {len(shared)}")

sorted_shared = sorted(shared.items(), key=lambda x: len(x[1]), reverse=True)
print("\n=== TOP SHARED ADDRESSES ===")
for addr, ents in sorted_shared[:10]:
    print(f"\nAddress: {addr[:80]}...")
    print(f"  Shared by {len(ents)} entities:")
    for e in ents:
        print(f"    - {e['name']} ({e['schema']})")

Addresses shared by 2+ entities: 3410

=== TOP SHARED ADDRESSES ===

Address: Iran...
  Shared by 216 entities:
    - Mohammad Mahdi Maghfoori (Person)
    - Fars News Agency (Company)
    - Spoločnosť Fajr Aviation Composite Industries (Company)
    - SAZEH MORAKAB CO. LTD (Company)
    - MARJAN METHANOL COMPANY (Company)
    - Ebrahim Shariatzadeh (Person)
    - Shahid Shoshtari Industries (Company)
    - Yusuf Ali MIRAJ (Person)
    - PARS MCS (Company)
    - AMMAR IBN YASIR BRIGADE (Company)
    - Tarabar-Gorous Trust Transportation (Company)
    - Itsec Team (Company)
    - fifteenth ocean gmbh & co. kg (Company)
    - IRANIAN PETROCHEMICAL INVESTMENT GROUP COMPANY (Company)
    - MOLOUDI, Mohammad (Person)
    - EGHTESAD NOVIN BANK (Company)
    - GHADIR SOLAR ELECTRICITY AND ENERGY (Company)
    - Pasar Gas Novin Trading Co (Company)
    - Ansar Bank Brokerage Company (Company)
    - Radiation Applications Development Co (Company)
    - Azar Pad Qeshm (Company)
    - Mansur Rava